In [0]:
# Verify files are in place
files = dbutils.fs.ls("/Volumes/shopflow/default/shopflow_row")
for f in files:
    print(f.name, f"\t{f.size:,} bytes")

In [0]:
BASE_PATH = "/Volumes/shopflow/default/shopflow_row/"

df_customers = spark.read.csv(
    BASE_PATH + "customers.csv",
    header= True,
    inferSchema = True
)

df_products = spark.read.csv(
    BASE_PATH + "products.csv",
    header= True,
    inferSchema = True
)

df_orders = spark.read.csv(
    BASE_PATH + "orders.csv",
    header=True,
    inferSchema=True
)

for name, df in [("customers", df_customers),
                 ("products",  df_products),
                 ("orders",    df_orders)]:
    print(f"\n=== {name} ({df.count():,} rows) ===")
    df.printSchema()

In [0]:
# Write customers → bronze.customers
df_customers.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("shopflow.bronze.customers")

# Write products → bronze.products
df_products.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("shopflow.bronze.products")

# Write orders → bronze.orders
df_orders.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("shopflow.bronze.orders")

print("✓ Bronze tables written successfully")

In [0]:
print("=== Row counts ===")
for tbl in ["shopflow.bronze.customers","shopflow.bronze.products","shopflow.bronze.orders"]:
    count = spark.table(tbl).count()
    print(f"  {tbl}: {count:,} rows")

print("\n=== Orders sample ===")
spark.table("shopflow.bronze.orders").show(5)

print("\n=== Delta history (orders) ===")
spark.sql("DESCRIBE HISTORY shopflow.bronze.orders").show(3, truncate=False)